In [ ]:
# reranker by lanceDB : https://www.youtube.com/watch?v=QMxCCxBYoaE

In [ ]:
from ollama import Client as OllamaClient
from app_vdb_milvus import MilvusHybridConfig, MilvusHybridVDB

ollama = OllamaClient(host="http://localhost:11434")

def emb_text(text: str) -> list[float]:
    return ollama.embeddings(model="qwen3-embedding:0.6b", prompt=text)["embedding"]

cfg = MilvusHybridConfig(
    uri="http://localhost:19530",
    collection="rag_minist_int_hybrid",
    embed_dim=1024,
    drop_if_exists=False,  # True pour recréer la collection
    analyzer_type="english", # à voir si embêtant avec des documents en français
)

vdb = MilvusHybridVDB(cfg, embed_fn=emb_text)

vdb.ensure_collection()

res = vdb.search_hybrid("Quelle est l'idée principale de DeepSeek-V2 ?", top_k=5)

for hit in res[0]:
    score = getattr(hit, "distance", None) or getattr(hit, "score", None)
    ent = getattr(hit, "entity", None)
    get = ent.get if hasattr(ent, "get") else lambda k, d=None: getattr(ent, k, d)
    print("score:", score, "|", get("id"), "| page", get("page_no"))
    print((get("text") or "")[:250], "\n---\n")


score: 0.03201844170689583 | data/240504434v5.pdf::s3::c3_4_14 | page 4
We construct a high-quality and multi-source pre-training corpus consisting of 8.1T tokens. Compared with the corpus used in DeepSeek 67B (our previous release) (DeepSeek-AI, 2024), this corpus features an extended amount of data, especially Chinese  
---

score: 0.016393441706895828 | pdfs/template_lightOn.docx::s34::c34_16_7 | page 16
##  Installation et configuration d'un Hive Metastore

Namespace Paradigm

K8S Cluster

L'architecture Paradigm offre une flexibilité totale, pouvant fonctionner sur un seul nœud ou se déployer sur plusieurs, tout en maintenant une communication séc 
---

score: 0.016129031777381897 | data/240504434v5.pdf::s1::c1_1_6 | page 1
## Abstract

We present DeepSeek-V2, a strong Mixture-of-Experts (MoE) language model characterized by economical training and efficient inference. It comprises 236B total parameters, of which 21B are activated for each token, and supports a context  
---

sco